# 5.6 - Prompt Optimization with DSPy

**Module 5 - Prompt Engineering** | Netsetos GenAI Engineering

This notebook stops hand-tuning prompt strings and starts *compiling* them: you declare a task as a typed DSPy signature, wrap it in a reasoning module, and (in the lesson's fenced examples) let an optimizer rewrite the wording and few-shot demos against a metric. The runnable cells here wire Gemini to DSPy and show the declare-the-task move end to end.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

A single reproducibility marker. The lesson uses seeded random values in a few places so that runs are repeatable, and this comment flags that intent at the top of the notebook.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

This is a bare comment cell - no imports, no execution, nothing to call. It exists to signal that the examples below are meant to be deterministic where randomness is involved.

**How the code works - step by step**
- **The one comment line** - a note that seeded random values are used throughout so results reproduce.

**In one line:** a reproducibility marker, not runnable logic.

**What you'll see:** (no output - environment prep)

## 1 - Wire up Gemini and point DSPy at it

Before you can compile a prompt you need two things running: a plain model call (for the LLM-as-judge metric later) and DSPy configured to drive that same model. This cell builds both on the current unified `google-genai` SDK - the older `google.generativeai` package was deprecated in 2025.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`).

In [ ]:
# pip install "google-genai>=1.0.0" dspy>=2.5
from google import genai
from google.genai import types
import os, dspy

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
MODEL = "gemini-3-flash"

def generate(prompt: str) -> str:
    try:
        return client.models.generate_content(model=MODEL, contents=prompt).text.strip()
    except Exception as e:
        return f"[API error: {e}]"

# Point DSPy at the same model - it builds and tunes prompts through this.
dspy.configure(lm=dspy.LM(f"gemini/{MODEL}", api_key=os.environ["GOOGLE_API_KEY"]))
print("ready")
# Output: ready

**Explanation**

This is the environment-prep cell: it creates a `genai.Client`, wraps a one-call `generate` helper you'll reuse all lesson, and then hands DSPy the same model so it can build and tune prompts underneath. Think of it as connecting one model to two front doors - a raw call and a DSPy-managed call.

**How the code works - step by step**
- **`from google import genai` / `types`, `import os, dspy`** - the unified Gemini SDK, plus DSPy.
- **`client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])`** - one client object; the deprecated package instead used a global `configure()` plus a per-call `GenerativeModel`.
- **`MODEL = "gemini-3-flash"`** - the model both paths target.
- **`generate(prompt)`** - a plain text-in/text-out call wrapped in `try/except` so a single network blip returns an error string instead of crashing a long run; this is the call the Step-2 judge uses.
- **`dspy.configure(lm=dspy.LM(f"gemini/{MODEL}", api_key=...))`** - points DSPy at the same model; `api_key` is passed explicitly because DSPy routes through LiteLLM, whose `gemini/` provider otherwise looks for `GEMINI_API_KEY`.

**In one line:** one model, two entry points - a raw `generate` and a DSPy-driven `LM`.

**What you'll see:** Prints `ready` once the client is built and DSPy is configured.

## 2 - Declare the task as a DSPy signature

Here is the whole DSPy idea in one runnable block: you describe *what* the task is (a typed signature), pick *how* it reasons (a module), and DSPy writes the actual prompt string for you. No prompt wording is authored by hand.

In [ ]:
import dspy

class Classify(dspy.Signature):
    """Classify a customer support ticket into one category."""
    ticket: str = dspy.InputField()
    category: str = dspy.OutputField(desc="Hardware, Software, Connectivity, Billing, or Account")

classify = dspy.ChainOfThought(Classify)      # module = HOW it reasons

result = classify(ticket="My laptop won't connect to the company VPN")
print(result.category)
# Output: Connectivity         (DSPy wrote the prompt; you wrote the SIGNATURE)

**Explanation**

This cell separates the task from the wording. `Classify` is a typed contract - inputs, outputs, and a one-line docstring - and `ChainOfThought` is the reasoning strategy wrapped around it. You supply the signature; DSPy assembles the instruction and format, and an optimizer could later rewrite that wording without you touching the task.

**How the code works - step by step**
- **`class Classify(dspy.Signature)`** - the *what*: a docstring describing the task, an input field `ticket`, and an output field `category` whose `desc` names the allowed labels (Hardware, Software, Connectivity, Billing, Account).
- **`dspy.ChainOfThought(Classify)`** - the *how*: wraps the signature in a think-then-answer strategy; swapping to `dspy.Predict` or `dspy.ReAct` changes the reasoning without editing the signature.
- **`classify(ticket="...VPN")`** - runs the module on one ticket; DSPy built and sent the prompt behind the scenes.
- **`result.category`** - reads the structured output field, not raw text.

**In one line:** declare the task (signature) + choose a strategy (module) -> DSPy writes the prompt.

**What you'll see:** Prints the predicted category - `Connectivity` for the VPN ticket - produced by a prompt DSPy generated from the signature.

Across these cells you connected a live Gemini model to DSPy and watched the core reframe in action: you write a typed *signature* (the task) and pick a *module* (the reasoning strategy), and DSPy generates the actual prompt string. That is the hook the rest of the lesson hangs on - once the task is declared and a metric exists, an optimizer like MIPROv2 or GEPA can search instructions and few-shot demos for you, and you ship the winner like code (A/B, regression, canary). Next stop is Module 14, where this prompt-level core grows into full eval-driven development.